In [1]:
"""
test_epoch_phoneme_tracker.py — Longitudinal Phoneme Evolution Tracker.
Evolves a language across 200 epochs and prints statistical snapshots every 25 epochs:
- Active Vowels (Oral & Nasalized)
- Active Consonants (Clicks, Ejectives, Implosives, Pharyngealized, Labialized, Palatalized)
- Consonant-to-Vowel (C:V) Ratio & Shannon Entropy
- Top Consonant & Vowel Frequency Percentages
"""

from __future__ import annotations

import math
from collections import Counter

from phoneme import Phoneme, PhonemeKind
from language import Language
from world_map import Environment
from articulatory_space import Airstream, SecondaryArticulation


def populate_proto_phonemes(lang: Language):
    # Cardinal Vowels
    for pt in [(6,0,0), (6,2,1), (4,0,0), (4,2,1), (0,1,0)]:  # /i, u, e, o, a/
        lang.add_phoneme(Phoneme(PhonemeKind.VOWEL, pt))
        
    # Baseline Oral Consonants (Strictly 2 Nasals: /m, n/)
    for pt in [
        (0,0,0), (0,0,1), (3,0,0), (3,0,1), (7,0,0), (7,0,1),   # p, b, t, d, k, g
        (1,4,0), (1,4,1), (3,4,0), (3,4,1), (4,4,0), (7,4,0), (10,4,0), # f, v, s, z, ʃ, x, h
        (0,1,1), (3,1,1),                                       # m, n
        (3,7,1), (0,7,1)                                        # l, w
    ]:
        lang.add_phoneme(Phoneme(PhonemeKind.CONSONANT, pt))


def print_epoch_phoneme_snapshot(lang: Language, epoch: int):
    stats = lang.get_phoneme_distribution_stats()
    
    # 1. Separate Vowels (Oral vs. Nasalized)
    v_all = sorted(list(set(p.ipa for p in lang.vowels)))
    v_oral = [v for v in v_all if "\u0303" not in v]
    v_nasal = [v for v in v_all if "\u0303" in v]

    # 2. Analyze Consonant Airstreams & Layers
    c_all = sorted(list(set(p.ipa for p in lang.consonants)))
    
    clicks = [c for c in c_all if any(sym in c for sym in ('ǃ', 'ʘ', 'ǂ', 'ǁ', 'ǀ'))]
    ejectives = [c for c in c_all if "ʼ" in c and not any(sym in c for sym in ('ǃ', 'ʘ', 'ǂ', 'ǁ', 'ǀ'))]
    implosives = [c for c in c_all if any(sym in c for sym in ('ɓ', 'ɗ', 'ɠ', 'ʄ', 'ʛ'))]
    pharyngealized = [c for c in c_all if "ˁ" in c]
    labialized = [c for c in c_all if "ʷ" in c]
    palatalized = [c for c in c_all if "ʲ" in c]
    aspirated = [c for c in c_all if "ʰ" in c or "ʱ" in c]

    # Sample word
    sample_words = list(lang.words.values())
    sample_w = sample_words[len(sample_words) // 2] if sample_words else None
    sample_str = f"/{sample_w.form}/ (#{sample_w.id})" if sample_w else "N/A"

    print("=" * 105)
    print(f"  EPOCH {epoch:>3} / 200 SNAPSHOT — {lang.name.upper()}")
    print("=" * 105)
    print(f"• Living Vocabulary  : {len(lang.words):<5} words | Total Phoneme Segments: {stats.get('raw_total_segments', 0)}")
    print(f"• Consonant/Vowel    : {stats.get('cv_ratio', 0.0):<5} | Shannon Entropy: {stats.get('entropy_evenness', 0.0):.3f} / 1.000")
    print(f"• Sample Living Word : {sample_str}")
    print("-" * 105)

    # Vowels Display
    v_nasal_str = f" | Nasalized ({len(v_nasal)}): {', '.join(v_nasal)}" if v_nasal else ""
    print(f"• Vowels ({len(v_all)} Total) : Oral ({len(v_oral)}): {', '.join(v_oral)}{v_nasal_str}")

    # Consonants Display
    print(f"• Consonants ({len(c_all)} Total): {', '.join(c_all)}")

    # Special Airstream / Layer Breakdown
    specials = []
    if clicks: specials.append(f"Clicks ({len(clicks)}): {', '.join(clicks)}")
    if ejectives: specials.append(f"Ejectives ({len(ejectives)}): {', '.join(ejectives)}")
    if implosives: specials.append(f"Implosives ({len(implosives)}): {', '.join(implosives)}")
    if pharyngealized: specials.append(f"Pharyngealized ({len(pharyngealized)}): {', '.join(pharyngealized)}")
    if labialized: specials.append(f"Labialized ({len(labialized)}): {', '.join(labialized)}")
    if palatalized: specials.append(f"Palatalized ({len(palatalized)}): {', '.join(palatalized)}")
    if aspirated: specials.append(f"Aspirated ({len(aspirated)}): {', '.join(aspirated)}")

    if specials:
        print("• Emergent Layers    :")
        for s in specials:
            print(f"  ↳ {s}")

    # Top Frequencies
    top_c = stats.get("top_5_consonants", [])
    top_v = stats.get("top_3_vowels", [])
    
    top_c_str = ", ".join([f"/{ipa}/ ({pct:.1f}%)" for ipa, pct in top_c])
    top_v_str = ", ".join([f"/{ipa}/ ({pct:.1f}%)" for ipa, pct in top_v])
    
    print(f"• Top Consonants     : {top_c_str}")
    print(f"• Top Vowels         : {top_v_str}")
    print("")


def run_epoch_phoneme_tracker():
    print("\n" + "#" * 105)
    print("  LONGITUDINAL 200-EPOCH PHONEME TRACKER (SNAPSHOTS EVERY 25 EPOCHS)")
    print("#" * 105 + "\n")

    # You can customize the environment here to test any biome (Savannah, Alpine, Jungle, Desert)
    env = Environment(temperature=0.52, humidity=0.55, altitude=0.25, vegetation=0.40, ambient_noise=0.1, population=0.7)

    lang = Language("Language", environment=env)
    populate_proto_phonemes(lang)
    lang.generate_lexicon()

    # Epoch 0 Snapshot (Baseline Proto-Language)
    print_epoch_phoneme_snapshot(lang, epoch=0)

    # Evolve 200 Generations with Snapshots every 25 Epochs
    for ep in range(1, 401):
        lang.evolve(reduction_strength=0.75, enable_neologisms=True, current_generation=ep)
        
        if ep % 25 == 0:
            print_epoch_phoneme_snapshot(lang, epoch=ep)

    print("#" * 105)
    print("  LONGITUDINAL TRACKER RUN COMPLETE!")
    print("#" * 105)


if __name__ == "__main__":
    run_epoch_phoneme_tracker()


#########################################################################################################
  LONGITUDINAL 200-EPOCH PHONEME TRACKER (SNAPSHOTS EVERY 25 EPOCHS)
#########################################################################################################

  EPOCH   0 / 200 SNAPSHOT — LANGUAGE
• Living Vocabulary  : 69    words | Total Phoneme Segments: 434
• Consonant/Vowel    : 1.81  | Shannon Entropy: 0.928 / 1.000
• Sample Living Word : /ˈtäkku/ (#1035)
---------------------------------------------------------------------------------------------------------
• Vowels (5 Total) : Oral (5): e, i, o, u, ä
• Consonants (15 Total): b, d, h, k, l, m, n, p, t, w, x, z, ɡ, ɦ, ʃ
• Top Consonants     : /ʃ/ (17.6%), /t/ (11.8%), /k/ (8.3%), /l/ (7.0%), /s/ (6.8%)
• Top Vowels         : /i/ (35.4%), /ä/ (22.0%), /u/ (19.3%)

  EPOCH  25 / 200 SNAPSHOT — LANGUAGE
• Living Vocabulary  : 105   words | Total Phoneme Segments: 439
• Consonant/Vowel    : 1.2   | Shannon Entr

In [1]:
"""
test_export_complete_dictionary.py — Comprehensive Linguistic Atlas, Lexicon Exporter & Syntactic Clause Generator.
Features:
- Live Syntactic Clause Showcase (Transitive, Ditransitive, Relative Clauses, DOM & Scrambling)
- Complete 2D Noun Declension Grids (with Case Syncretism & Nominative Typology)
- Verb Conjugation Atlas (Ablaut, Transfixation, Reduplication, Suppletion)
- Somatic Numeral Counting Chains, Polysemy Networks, and Root Genealogies
"""

from __future__ import annotations

import os
from collections import defaultdict

from phoneme import Phoneme, PhonemeKind
from language import Language
from world_map import Environment
from grammar import GrammarEngine, NounPhrase, Clause, ClauseValency
from semantics import SemanticVector, SemanticSpace, DerivationType
from word import Word, LexicalCategory
from translator import SemanticTranslator

n_epochs = 600


def print_header(title: str):
    print(f"\n{'=' * 115}\n  {title}\n{'=' * 115}")


def populate_proto_phonemes(lang: Language):
    # Cardinal Vowel Grid (/i, u, e, o, a/) with Low Central Anchor (0, 1, 0)
    for pt in [(6,0,0), (6,2,1), (4,0,0), (4,2,1), (0,1,0)]:
        lang.add_phoneme(Phoneme(PhonemeKind.VOWEL, pt))
    # Standard Oral Consonants
    for pt in [
        (0,0,0), (0,0,1), (3,0,0), (3,0,1), (7,0,0), (7,0,1),
        (1,4,0), (1,4,1), (3,4,0), (3,4,1), (4,4,0), (7,4,0), (10,4,0),
        (0,1,1), (3,1,1), (3,7,1), (0,7,1)
    ]:
        lang.add_phoneme(Phoneme(PhonemeKind.CONSONANT, pt))


def export_complete_dictionary(lang: Language, current_epoch: int, output_filename: str | None = None) -> str:
    filename = output_filename or f"dictionary_{lang.name.replace(' ', '_')}_epoch_{current_epoch}.txt"
    paradigm = GrammarEngine.discover_grammar_system(lang)
    alignment = GrammarEngine.get_alignment_system(lang)
    s_idx = lang.profile.synthesis_index
    h = lang.profile.head_directionality

    words = sorted(list(lang.words.values()), key=lambda w: w.plain_form)
    total_words = len(words)
    poly_words = [w for w in words if len(w.senses) > 1]
    multi_morpheme_words = [w for w in words if len(w.morphemes) >= 2]

    v_raw = list(set(p.ipa for p in lang.vowels))
    oral_vowels = sorted([v for v in v_raw if "\u0303" not in v])
    nasal_vowels = sorted([v for v in v_raw if "\u0303" in v])
    consonants = sorted(list(set(p.ipa for p in lang.consonants)))

    lines = []
    
    # =========================================================================
    # 1. HEADER & METADATA
    # =========================================================================
    lines.append("=" * 115)
    lines.append(f"  EVOLANG COMPLETE ETYMOLOGICAL DICTIONARY & LINGUISTIC ATLAS")
    lines.append(f"  Language: {lang.name} | Epoch / Generation: {current_epoch}")
    lines.append("=" * 115)
    lines.append(f"• Total Living Words  : {total_words} words ({len(poly_words)} polysemous entries with extended senses)")
    lines.append(f"• Multi-Morphemic     : {len(multi_morpheme_words)} compound/affixed words ({len(multi_morpheme_words)/total_words*100:.1f}%)")
    lines.append(f"• Phonemic Inventory  : {len(oral_vowels)} Oral Vowels" + (f", {len(nasal_vowels)} Nasal Vowels" if nasal_vowels else "") + f", {len(consonants)} Consonants")
    lines.append(f"• Core Oral Vowels    : {', '.join(oral_vowels)}")
    if nasal_vowels:
        lines.append(f"• Nasalized Vowels    : {', '.join(nasal_vowels)}")
    lines.append(f"• Consonant Inventory : {', '.join(consonants)}")
    lines.append(f"• Morphosyntax        : Word Order: {lang.word_order} (H={h:.3f}) | Synthesis: {s_idx:.3f}")
    lines.append(f"• Alignment System    : {alignment}")
    
    n_classes_count = len(paradigm.noun_classes)
    lines.append(f"• Emergent Classes    : {n_classes_count} Noun Classes Discovered" if n_classes_count > 0 else "• Emergent Classes    : 0 Classes (Genderless / Classless Language)")
    lines.append(f"• Verb Conjugations   : {len(paradigm.verb_classes)} Verbal Classes Discovered" if paradigm.verb_classes else "• Verb Conjugations   : 0 Classes (Isolating Bare Verbs)")
    lines.append(f"• Number System       : {paradigm.number_system}")
    lines.append("")

    # =========================================================================
    # 2. MASTER GRAMMATICAL ATLAS GRIDS
    # =========================================================================
    lines.append("=" * 115)
    lines.append("  SECTION 1: MASTER GRAMMATICAL ATLAS & INFLECTIONAL STRATEGY MATRIX")
    lines.append("=" * 115)
    
    # TABLE 1: MASTER 2D NOUN DECLENSION MATRIX
    lines.append(f"TABLE 1: MASTER 2D NOUN DECLENSION MATRIX ({paradigm.number_system})")
    lines.append("-" * 115)

    sample_content = [w for w in words if GrammarEngine.get_part_of_speech(w) == "Noun" and len(w.phonemes) >= 3]
    sample_ref = sample_content[0] if sample_content else words[0]
    sample_decl = GrammarEngine.get_noun_declension_table(sample_ref, lang, paradigm)
    case_keys = list(sample_decl.keys())

    # Nominative Strategy Label
    if s_idx >= 0.50:
        nom_strategy_desc = "Overt Fused Nominative Ending (Latin -us / Greek -os model)"
    elif h < 0.35 and paradigm.spatial_operators:
        nom_strategy_desc = "Postpositional Subject Clitic (Japanese -ga / -wa model)"
    else:
        nom_strategy_desc = "Zero-Marked Bare Root (English / Mandarin Ø-Nominative model)"

    lines.append(f"• Nominative Case Strategy : [{nom_strategy_desc}]")
    lines.append(f"• Direct Object Strategy   : [Differential Object Marking: Animate marked, Inanimate Ø in SVO]")
    lines.append("-" * 115)

    if paradigm.noun_classes:
        for n_cls in paradigm.noun_classes:
            content_nouns = [
                w for w in words 
                if GrammarEngine.get_part_of_speech(w) == "Noun" 
                and len(w.phonemes) >= 3 
                and GrammarEngine.classify_noun_dynamically(w, paradigm, lang) == n_cls
            ]
            
            sample_noun = content_nouns[0] if content_nouns else sample_ref
            decl = GrammarEngine.get_noun_declension_table(sample_noun, lang, paradigm)
            cls_marker_str = f"/{n_cls.marker.form}/" if n_cls.marker else "Ø"

            # Plural Strategy Diagnostic
            _, is_umlaut = GrammarEngine.apply_vocalic_ablaut(sample_noun, mode="plural", lang=lang)
            if is_umlaut and (sample_noun.usage_frequency >= 2.0 or s_idx >= 0.40):
                plural_strategy = "Vocalic Umlaut (Germanic i-mutation: man -> men)"
            elif "pl" in paradigm.number_operators and s_idx >= 0.30:
                plural_strategy = f"Concatenative Suffixation (+/{paradigm.number_operators['pl'].form}/)"
            else:
                plural_strategy = "Invariant Stem / Analytic Plural Particle"

            noun_meaning = SemanticTranslator.translate(sample_noun, lang)
            lines.append(f"• {n_cls.label} [Class Concord Prefix: {cls_marker_str}]")
            lines.append(f"  ↳ Sample Root     : /{sample_noun.plain_form}/ «{noun_meaning}» (ID #{sample_noun.id})")
            lines.append(f"  ↳ Plural Strategy : [{plural_strategy}]")
            decl_strs = [f"{k}: {decl.get(k, '-')}" for k in case_keys]
            lines.append("  ↳ Paradigms       : " + " | ".join(decl_strs))
            lines.append("-" * 115)
    else:
        lines.append("• Universal Noun Declension (Genderless / Classless Language):")
        decl_strs = [f"{k}: {sample_decl.get(k, '-')}" for k in case_keys]
        lines.append("  ↳ " + " | ".join(decl_strs))
        lines.append("-" * 115)
    lines.append("")

    # TABLE 2: MASTER VERB CONJUGATION ATLAS
    lines.append("TABLE 2: MASTER VERB CONJUGATION ATLAS (Aktionsart Tracks × TAM × Morphological Strategy)")
    lines.append("-" * 115)

    if paradigm.verb_classes:
        for v_cls in paradigm.verb_classes:
            content_verbs = [
                w for w in words 
                if GrammarEngine.get_part_of_speech(w) == "Verb" 
                and len(w.phonemes) >= 3 
                and GrammarEngine.classify_verb_dynamically(w, paradigm, lang).class_id == v_cls.class_id
            ]
            
            sample_verb = content_verbs[0] if content_verbs else next((w for w in words if w.vector.dynamism >= 0.50), words[0])
            v_conj = GrammarEngine.get_verb_conjugation_table(sample_verb, lang, paradigm)
            theme_str = f"/{v_cls.thematic_marker.form}/" if v_cls.thematic_marker else "Ø"

            # Past / Aspect Strategy Diagnostic
            if v_cls.morph_strategy == "transfix":
                past_strategy = "Semitic Triconsonantal Root-and-Pattern Transfixation (k-t-b -> kataba)"
            elif v_cls.morph_strategy == "ablaut" or sample_verb.usage_frequency >= 2.0:
                past_strategy = "Internal Vocalic Ablaut (Indo-European apophony: sing -> sang)"
            elif v_cls.morph_strategy == "reduplication":
                past_strategy = "Iconic Partial Reduplication (Tagalog/Latin: su-sulat, pe-puli)"
            else:
                t_op = paradigm.temporal_operators[0] if paradigm.temporal_operators else None
                op_str = f"+/{t_op.form}/" if t_op else "Free Particle"
                past_strategy = f"Concatenative TAM Affixation ({op_str})"

            verb_meaning = SemanticTranslator.translate(sample_verb, lang)
            lines.append(f"• {v_cls.label} [Thematic Stem: {theme_str} | Aspect Track: {v_cls.aspect_label}]")
            lines.append(f"  ↳ Sample Root     : /{sample_verb.plain_form}/ «{verb_meaning}» (ID #{sample_verb.id})")
            lines.append(f"  ↳ Past Strategy   : [{past_strategy}]")
            
            conj_strs = [f"{k}: {v}" for k, v in v_conj.items()]
            lines.append("  ↳ Conjugations    : " + " | ".join(conj_strs[:5]))
            lines.append("  ↳ Past Tenses     : " + " | ".join(conj_strs[5:]))
            lines.append("-" * 115)
    lines.append("")

    # TABLE 3: MASTER SOMATIC NUMERAL COUNTING ATLAS
    lines.append(f"TABLE 3: MASTER SOMATIC NUMERAL COUNTING ATLAS ({paradigm.numeral_chain.name})")
    lines.append("-" * 115)
    
    chain = paradigm.numeral_chain
    lines.append(f"• Somatic Hierarchy: Sub-Base: {chain.sub_base or 'None'} | Primary Base: {chain.primary_base} | Super-Base: {chain.super_base or 'None'}")
    sample_ints = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 20, 25, 40, 50, 60, 73, 100]
    numeral_strs = [f"{n}: {GrammarEngine.format_numeral(n, lang, paradigm)}" for n in sample_ints]
    lines.append("  ↳ " + " | ".join(numeral_strs[:7]))
    lines.append("  ↳ " + " | ".join(numeral_strs[7:14]))
    lines.append("  ↳ " + " | ".join(numeral_strs[14:]))
    lines.append("")
    
    # =========================================================================
    # 3. CLOSED-CLASS OPERATOR INVENTORY
    # =========================================================================
    lines.append("-" * 115)
    lines.append("  SECTION 2: RECRUITED CLOSED-CLASS OPERATOR INVENTORY (COMPACT CORE MORAS)")
    lines.append("-" * 115)
    
    lines.append(f"A. Tense, Aspect & Temporal Operators ({len(paradigm.temporal_operators)}):")
    for op in paradigm.temporal_operators:
        lines.append(f"   - #{op.id:<5} /{op.form}/  (U={op.usage_frequency:.2f}) -> Meaning: {SemanticTranslator.translate(op, lang)}")
    
    lines.append(f"\nB. Spatial Relators & Case Operators ({len(paradigm.spatial_operators)}):")
    for idx, op in enumerate(paradigm.spatial_operators, 1):
        role_label = "Acc/Erg" if idx == 1 else ("Dat" if idx == 2 else "Loc/Abl")
        lines.append(f"   - #{op.id:<5} /{op.form}/  (U={op.usage_frequency:.2f}) -> [{role_label}] Case Operator ({SemanticTranslator.translate(op, lang)})")

    lines.append(f"\nC. Modals & Negators ({len(paradigm.modal_operators)}):")
    for op in paradigm.modal_operators:
        lines.append(f"   - #{op.id:<5} /{op.form}/  (U={op.usage_frequency:.2f}) -> Meaning: {SemanticTranslator.translate(op, lang)}")

    if paradigm.noun_classes:
        lines.append(f"\nD. Emergent Noun Class Concord Markers ({len(paradigm.noun_classes)} Classes):")
        for n_cls in paradigm.noun_classes:
            m_str = f"/{n_cls.marker.form}/ (#{n_cls.marker.id})" if n_cls.marker else "Ø"
            lines.append(f"   - {n_cls.label:<40} -> Marker: {m_str}")

    if paradigm.pronouns:
        lines.append(f"\nE. Recruited Personal Pronouns (Monosyllabic Cores):")
        for p_label, p_word in paradigm.pronouns.items():
            lines.append(f"   - {p_label:<6} : #{p_word.id:<5} /{p_word.form}/")

    lines.append(f"\nF. Subordinating Relativizers [RC] & Complementizers [CP]:")
    for op in paradigm.relativizers:
        lines.append(f"   - #{op.id:<5} /{op.form}/  [REL]")
    for op in paradigm.complementizers:
        lines.append(f"   - #{op.id:<5} /{op.form}/  [COMP]")
    lines.append("")

    lines.append(f"G. Emergent Definiteness & Article System: [{paradigm.article_system}]")
    if "def" in paradigm.articles:
        op = paradigm.articles["def"]
        lines.append(f"   - Definite Article   [DEF]   : #{op.id:<5} /{op.form}/  (Source: {SemanticTranslator.translate(op, lang)})")
    if "indef" in paradigm.articles:
        op = paradigm.articles["indef"]
        lines.append(f"   - Indefinite Article [INDEF] : #{op.id:<5} /{op.form}/  (Source: {SemanticTranslator.translate(op, lang)})")
    if not paradigm.articles:
        lines.append(f"   - (No grammatical articles; definiteness inferred from context)")
    lines.append("")

    # =========================================================================
    # 4. COMPLETE LIVING LEXICON ENTRIES
    # =========================================================================
    lines.append("=" * 115)
    lines.append("  SECTION 3: COMPLETE LIVING LEXICON (CANONICAL LEMMAS & PARADIGM STRATEGIES)")
    lines.append("=" * 115)

    for idx, w in enumerate(words, 1):
        c, a, v, p, d, s, e = w.vector.coords
        
        primary_meaning = SemanticTranslator.translate(w, lang)
        lemma_form, lemma_desc = GrammarEngine.render_lemma(w, lang, paradigm)
        pos = GrammarEngine.get_part_of_speech(w)
        
        cat_str = "FUNCTIONAL (Closed)" if w.category == LexicalCategory.FUNCTIONAL_CLOSED else f"CONTENT (Open, {pos})"
        syl_str = ".".join(syl.form for syl in w.syllables) if w.syllables else w.form

        if w.morphemes:
            m_list = [f"/{m.ipa}/ (Fam#{m.root_family_id}, {m.stage.value})" for m in w.morphemes]
            morph_str = " + ".join(m_list)
        else:
            morph_str = f"/{w.plain_form}/ (Root)"

        deriv = w.derivation
        if deriv.derivation_type == DerivationType.ROOT:
            if w.generation_born == 0:
                etym_str = f"Primordial Proto-Root (Family #{deriv.root_family_id}, Born Gen 0)"
            else:
                etym_str = f"Neologism / De Novo Root (Family #{deriv.root_family_id}, Born Gen {w.generation_born})"
        elif deriv.derivation_type == DerivationType.COMPOUND:
            parents = ", ".join(f"#{pid}" for pid in deriv.local_parent_ids)
            etym_str = f"Compound of [{parents}] (Family #{deriv.root_family_id}, Born Gen {w.generation_born})"
        elif deriv.derivation_type == DerivationType.SEMANTIC_SPLIT:
            parents = ", ".join(f"#{pid}" for pid in deriv.local_parent_ids)
            etym_str = f"Semantic Mitosis Split from [{parents}] (Family #{deriv.root_family_id}, Born Gen {w.generation_born})"
        elif deriv.derivation_type == DerivationType.AFFIXATION:
            parents = ", ".join(f"#{pid}" for pid in deriv.local_parent_ids)
            etym_str = f"Derivational Affixation of [{parents}] (Family #{deriv.root_family_id}, Born Gen {w.generation_born})"
        elif deriv.derivation_type == DerivationType.LOANWORD and deriv.provenance:
            prov = deriv.provenance
            etym_str = f"Loanword borrowed from '{prov.source_language}' (Donor #{prov.donor_word_id} /{prov.donor_form_at_borrowing}/ at Gen {prov.generation_borrowed})"
        else:
            etym_str = f"{deriv.derivation_type.value.upper()} (Family #{deriv.root_family_id})"

        lines.append(f"[{idx:>4}] #{w.id:<5}  /{lemma_form}/  [Lemma: {lemma_desc}]")
        lines.append(f"      • Definition   : {primary_meaning}")
        lines.append(f"      • Stem Phonetics: /{w.form}/ (Syllables: [{syl_str}])")
        lines.append(f"      • Typology     : {cat_str} | Frequency: {w.usage_frequency:.2f} | Age: {current_epoch - w.generation_born} gens")

        if pos == "Noun":
            decl_cases = GrammarEngine.get_noun_declension_table(w, lang, paradigm)
            decl_strs = [f"{case_name}: {case_form}" for case_name, case_form in decl_cases.items()]
            lines.append(f"      • Declension   : " + ", ".join(decl_strs))

        elif pos == "Verb":
            conj_table = GrammarEngine.get_verb_conjugation_table(w, lang, paradigm)
            conj_strs = [f"{aspect_name}: {form_val}" for aspect_name, form_val in conj_table.items()]
            lines.append(f"      • Conjugation  : " + ", ".join(conj_strs))

        elif pos == "Adjective":
            adj_concord = GrammarEngine.get_adjective_concord_table(w, lang, paradigm)
            concord_strs = [f"{cls_name}: {form_val}" for cls_name, form_val in adj_concord.items()]
            lines.append(f"      • Class Concord: " + ", ".join(concord_strs))

        lines.append(f"      • 7D Vector    : (C:{c:.2f}, A:{a:.2f}, V:{v:.2f}, P:{p:.2f}, D:{d:.2f}, S:{s:.2f}, E:{e:.2f})")
        lines.append(f"      • Morphology   : {morph_str}")
        lines.append(f"      • Etymology    : {etym_str}")

        poly_senses = SemanticTranslator.translate_polysemy(w, lang)
        if len(poly_senses) > 1:
            lines.append(f"      • Polysemy Network ({len(poly_senses)} Senses):")
            for s_idx, (s_meaning, pathway_tag, s_vec) in enumerate(poly_senses, 1):
                sc, sa, sv, sp, sd, ss, se = s_vec.coords
                lines.append(f"        ({s_idx}) [{pathway_tag}] «{s_meaning}» | (C:{sc:.2f}, A:{sa:.2f}, V:{sv:.2f}, P:{sp:.2f}, D:{sd:.2f}, S:{ss:.2f}, E:{se:.2f})")
        
        lines.append("")

    # =========================================================================
    # 5. ETYMOLOGICAL ROOT FAMILY INDEX
    # =========================================================================
    lines.append("=" * 115)
    lines.append(f"  SECTION 4: ETYMOLOGICAL ROOT FAMILY GENEALOGY INDEX")
    lines.append("=" * 115)

    families = defaultdict(list)
    for w in words:
        families[w.derivation.root_family_id].append(w)

    sorted_families = sorted(families.items(), key=lambda item: len(item[1]), reverse=True)

    for fam_id, fam_words in sorted_families:
        root_ancestor = next((w for w in fam_words if w.derivation.derivation_type == DerivationType.ROOT), fam_words[0])
        ancestor_meaning = SemanticTranslator.translate(root_ancestor, lang)
        lines.append(f"• ROOT FAMILY #{fam_id:<4} (Ancestor: /{root_ancestor.form}/ #{root_ancestor.id} «{ancestor_meaning}» | {len(fam_words)} living descendants):")
        descendant_strs = [f"#{w.id} /{w.form}/ «{SemanticTranslator.translate(w, lang)}» ({w.derivation.derivation_type.value})" for w in fam_words]
        lines.append(f"  ↳ " + ", ".join(descendant_strs))
        lines.append("")

    # =========================================================================
    # 6. LIVE SYNTACTIC SAMPLE SENTENCES & CLAUSE GENERATOR
    # =========================================================================
    lines.append("=" * 115)
    lines.append(f"  SECTION 5: LIVE SYNTACTIC SAMPLE SENTENCES & DISCOURSE SHOWCASE")
    lines.append("=" * 115)

    # Select the most entrenched, highest-frequency words for natural sentence demonstration
    def get_core_word(predicate_fn, default_idx):
        matching = [w for w in words if predicate_fn(w)]
        if matching:
            return max(matching, key=lambda w: w.usage_frequency)
        return words[default_idx]

    person_noun = get_core_word(lambda w: w.vector.distance_to(SemanticSpace.PERSON) < 0.35 and GrammarEngine.get_part_of_speech(w) == "Noun", 0)
    wolf_noun   = get_core_word(lambda w: w.vector.distance_to(SemanticSpace.BEAST) < 0.35 and GrammarEngine.get_part_of_speech(w) == "Noun", 1)
    child_noun  = get_core_word(lambda w: w.vector.distance_to(SemanticSpace.MOTHER) < 0.35 and GrammarEngine.get_part_of_speech(w) == "Noun", 2)
    stone_noun  = get_core_word(lambda w: w.vector.distance_to(SemanticSpace.STONE) < 0.35 and GrammarEngine.get_part_of_speech(w) == "Noun", 3)
    
    walk_verb  = get_core_word(lambda w: w.vector.dynamism >= 0.60 and w.vector.potency <= 0.50 and GrammarEngine.get_part_of_speech(w) == "Verb", 4)
    hunt_verb  = get_core_word(lambda w: w.vector.dynamism >= 0.70 and w.vector.potency >= 0.65 and GrammarEngine.get_part_of_speech(w) == "Verb", 5)
    give_verb  = get_core_word(lambda w: w.vector.sociality >= 0.60 and w.vector.dynamism >= 0.50 and GrammarEngine.get_part_of_speech(w) == "Verb", 6)

    # A. Intransitive Clause (1-Arg: Subject + Motion Verb)
    cl_intrans = GrammarEngine.assemble_complex_clause(
        lang=lang,
        subject=NounPhrase(head_noun=person_noun),
        predicate=walk_verb,
    )
    ipa_in, gl_in = cl_intrans.render(lang)
    lines.append("A. INTRANSITIVE CLAUSE (1-Arg: Subject + Predicate)")
    lines.append(f"   • Context    : \"The person ({SemanticTranslator.translate(person_noun, lang)}) walks ({SemanticTranslator.translate(walk_verb, lang)})\"")
    lines.append(f"   • Surface IPA: /{ipa_in}/")
    lines.append(f"   • Interlinear: {gl_in}\n")

    # B. Transitive Clause with Differential Object Marking (2-Arg: Animate Subject + Animate Object)
    cl_trans_anim = GrammarEngine.assemble_complex_clause(
        lang=lang,
        subject=NounPhrase(head_noun=wolf_noun),
        predicate=hunt_verb,
        direct_object=NounPhrase(head_noun=person_noun),
    )
    ipa_ta, gl_ta = cl_trans_anim.render(lang)
    lines.append("B. TRANSITIVE CLAUSE [ANIMATE OBJECT] (Differential Object Marking)")
    lines.append(f"   • Context    : \"The wolf ({SemanticTranslator.translate(wolf_noun, lang)}) hunts ({SemanticTranslator.translate(hunt_verb, lang)}) the person ({SemanticTranslator.translate(person_noun, lang)})\"")
    lines.append(f"   • Surface IPA: /{ipa_ta}/")
    lines.append(f"   • Interlinear: {gl_ta}\n")

    # C. Transitive Clause with Inanimate Object (Demonstrates Zero-Marking Ø in SVO)
    cl_trans_inan = GrammarEngine.assemble_complex_clause(
        lang=lang,
        subject=NounPhrase(head_noun=person_noun),
        predicate=hunt_verb,
        direct_object=NounPhrase(head_noun=stone_noun),
    )
    ipa_ti, gl_ti = cl_trans_inan.render(lang)
    lines.append("C. TRANSITIVE CLAUSE [INANIMATE OBJECT] (Demonstrating Accusative Zero-Marking Ø)")
    lines.append(f"   • Context    : \"The person strikes the stone ({SemanticTranslator.translate(stone_noun, lang)})\"")
    lines.append(f"   • Surface IPA: /{ipa_ti}/")
    lines.append(f"   • Interlinear: {gl_ti}\n")

    # D. Ditransitive Clause (3-Arg: Agent + Recipient + Theme)
    cl_ditrans = GrammarEngine.assemble_complex_clause(
        lang=lang,
        subject=NounPhrase(head_noun=person_noun),
        predicate=give_verb,
        direct_object=NounPhrase(head_noun=stone_noun),
        indirect_object=NounPhrase(head_noun=child_noun),
    )
    ipa_di, gl_di = cl_ditrans.render(lang)
    lines.append("D. DITRANSITIVE RECIPIENT CLAUSE (Dative / Case Syncretism)")
    lines.append(f"   • Context    : \"The person gives stone to the child ({SemanticTranslator.translate(child_noun, lang)})\"")
    lines.append(f"   • Surface IPA: /{ipa_di}/")
    lines.append(f"   • Interlinear: {gl_di}\n")

    # E. Complex Subordinated Relative Clause
    rel_op = paradigm.relativizers[0] if paradigm.relativizers else None
    rc_clause = Clause(
        predicate=walk_verb,
        valency=ClauseValency.INTRANSITIVE,
        subject=NounPhrase(head_noun=person_noun),
        relativizer=rel_op,
        is_subordinate=True
    )
    complex_subj = NounPhrase(head_noun=person_noun, relative_clause=rc_clause)
    cl_subord = GrammarEngine.assemble_complex_clause(
        lang=lang,
        subject=complex_subj,
        predicate=hunt_verb,
        direct_object=NounPhrase(head_noun=wolf_noun),
    )
    ipa_sub, gl_sub = cl_subord.render(lang)
    lines.append("E. COMPLEX SUBORDINATED RELATIVE CLAUSE [RC]")
    lines.append(f"   • Context    : \"The person [who walks] hunts the wolf\"")
    lines.append(f"   • Surface IPA: /{ipa_sub}/")
    lines.append(f"   • Interlinear: {gl_sub}\n")

    full_text = "\n".join(lines)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(full_text)

    return filename


def run_dictionary_export_test():
    print_header(f"1. SIMULATING {n_epochs} GENERATIONS OF A LIVING SPEECH COMMUNITY")

    env = Environment(temperature=0.52, humidity=0.6, altitude=0.3, vegetation=0.4, population=0.9, ambient_noise=0.2)
    lang = Language("Yamato-Metropolis", environment=env)
    populate_proto_phonemes(lang)
    lang.generate_lexicon()

    print(f"Language: {lang.name}")
    print(f"Environment: T={env.temperature:.2f}, H={env.humidity:.2f}, Alt={env.altitude:.2f}, Pop={env.population:.2f}")
    print(f"Initial Vocabulary: {len(lang.words)} proto-words\n")

    print(f"Evolving across {n_epochs} generational epochs...")
    for ep in range(1, n_epochs+1):
        lang.evolve(reduction_strength=0.75, enable_neologisms=True, current_generation=ep)
        if ep % 25 == 0 or ep == 1:
            print(f"  --> Epoch {ep:>3} / {n_epochs} Complete | Lexicon: {len(lang.words):<5} words | Active Phonemes: {len(lang.phonemes)}P")

    print_header("2. EXPORTING COMPLETE LEXICON & ETYMOLOGICAL DICTIONARY")

    saved_file = export_complete_dictionary(lang, current_epoch=n_epochs)
    file_size_kb = os.path.getsize(saved_file) / 1024.0

    print(f"✓ COMPLETE DICTIONARY & LINGUISTIC ATLAS SUCCESSFULLY EXPORTED TO DISK!")
    print(f"  • Filename   : {saved_file}")
    print(f"  • File Size  : {file_size_kb:.2f} KB")
    print(f"  • Total Words: {len(lang.words)} entries fully glossed with Declensions, Conjugations, Concord & Syntax.")
    print(f"\nYou can now open '{saved_file}' in any text editor to inspect every word and grammatical paradigm!")


if __name__ == "__main__":
    run_dictionary_export_test()


  1. SIMULATING 600 GENERATIONS OF A LIVING SPEECH COMMUNITY
Language: Yamato-Metropolis
Environment: T=0.52, H=0.60, Alt=0.30, Pop=0.90
Initial Vocabulary: 75 proto-words

Evolving across 600 generational epochs...
  --> Epoch   1 / 600 Complete | Lexicon: 78    words | Active Phonemes: 16P
  --> Epoch  25 / 600 Complete | Lexicon: 113   words | Active Phonemes: 20P
  --> Epoch  50 / 600 Complete | Lexicon: 172   words | Active Phonemes: 21P
  --> Epoch  75 / 600 Complete | Lexicon: 263   words | Active Phonemes: 19P
  --> Epoch 100 / 600 Complete | Lexicon: 463   words | Active Phonemes: 24P
  --> Epoch 125 / 600 Complete | Lexicon: 843   words | Active Phonemes: 22P
  --> Epoch 150 / 600 Complete | Lexicon: 1490  words | Active Phonemes: 24P
  --> Epoch 175 / 600 Complete | Lexicon: 1787  words | Active Phonemes: 24P
  --> Epoch 200 / 600 Complete | Lexicon: 1915  words | Active Phonemes: 24P
  --> Epoch 225 / 600 Complete | Lexicon: 1948  words | Active Phonemes: 24P
  --> Epoch 2

In [4]:
from pronouncer import Pronouncer, IPANormalizer

text = "ˈɰeaʃʼsɰ"
Pronouncer.speak(text)

"'weaSsw"

In [1]:
"""
test_phoneme_statistics.py — Deep statistical audit of phoneme distributions across diverse biomes.
Checks for over-represented sounds, markedness balance, Shannon entropy, and C:V ratios across 200 epochs.
"""

from __future__ import annotations

import math
from phoneme import Phoneme, PhonemeKind
from language import Language
from world_map import Environment


def populate_proto_phonemes(lang: Language):
    """Initializes standard cardinal vowels and baseline consonants."""
    # Cardinal Vowels
    for pt in [(6, 0, 0), (6, 2, 1), (4, 0, 0), (4, 2, 1), (0, 1, 0)]:  # /i, u, e, o, a/
        lang.add_phoneme(Phoneme(PhonemeKind.VOWEL, pt))

    # Baseline Oral Consonants
    for pt in [
        (0, 0, 0), (0, 0, 1), (3, 0, 0), (3, 0, 1), (7, 0, 0), (7, 0, 1),   # p, b, t, d, k, g
        (1, 4, 0), (1, 4, 1), (3, 4, 0), (3, 4, 1), (4, 4, 0), (7, 4, 0), (10, 4, 0),  # f, v, s, z, ʃ, x, h
        (0, 1, 1), (3, 1, 1),                                                 # m, n
        (3, 7, 1), (0, 7, 1)                                                  # l, w
    ]:
        lang.add_phoneme(Phoneme(PhonemeKind.CONSONANT, pt))


def analyze_anomalies(stats: dict, total_words: int) -> list[str]:
    """Detects statistical imbalances, single-sound monopolies, and phonotactic anomalies."""
    alerts = []
    
    # 1. Consonant Monopoly Check (No single consonant should exceed 14%)
    for ipa, pct in stats.get("consonants", {}).items():
        if pct > 14.0:
            alerts.append(f"Consonant Monopoly: /{ipa}/ ({pct:.1f}%)")

    # 2. Vowel Monopoly Check (No single vowel should exceed 38%)
    for ipa, pct in stats.get("vowels", {}).items():
        if pct > 38.0:
            alerts.append(f"Vowel Monopoly: /{ipa}/ ({pct:.1f}%)")

    # 3. C:V Ratio Bounds
    cv = stats.get("cv_ratio", 1.5)
    if cv < 0.65:
        alerts.append(f"Abnormal Low C:V Ratio ({cv:.2f} < 0.65)")
    elif cv > 2.60:
        alerts.append(f"Abnormal High C:V Ratio ({cv:.2f} > 2.60)")

    # 4. Average Word Length Creep
    raw_segments = stats.get("raw_total_segments", 0)
    avg_len = raw_segments / max(1, total_words)
    if avg_len > 7.5:
        alerts.append(f"Word Length Bloat (Mean {avg_len:.1f} segments/word > 7.5)")
    elif avg_len < 3.5:
        alerts.append(f"Excessive Word Compression (Mean {avg_len:.1f} segments/word < 3.5)")

    # 5. Shannon Entropy Threshold
    entropy = stats.get("entropy_evenness", 1.0)
    if entropy < 0.68:
        alerts.append(f"Low Shannon Entropy ({entropy:.3f} < 0.68)")

    return alerts


def print_language_stats(lang: Language):
    stats = lang.get_phoneme_distribution_stats()
    total_words = len(lang.words)
    raw_segments = stats.get("raw_total_segments", 0)
    avg_len = raw_segments / max(1, total_words)
    anomalies = analyze_anomalies(stats, total_words)

    c_all = sorted(list(set(p.ipa for p in lang.consonants)))
    v_all = sorted(list(set(p.ipa for p in lang.vowels)))

    # Detect emergent non-pulmonic layers
    clicks = [c for c in c_all if any(sym in c for sym in ('ǃ', 'ʘ', 'ǂ', 'ǁ', 'ǀ'))]
    ejectives = [c for c in c_all if "ʼ" in c and not any(sym in c for sym in ('ǃ', 'ʘ', 'ǂ', 'ǁ', 'ǀ'))]
    implosives = [c for c in c_all if any(sym in c for sym in ('ɓ', 'ɗ', 'ɠ', 'ʄ', 'ʛ'))]
    pharyngealized = [c for c in c_all if "ˁ" in c]

    sample_words = list(lang.words.values())
    sample_w = sample_words[len(sample_words) // 2] if sample_words else None
    sample_str = f"/{sample_w.form}/ (#{sample_w.id})" if sample_w else "N/A"

    print(f"\n{'='*45} {lang.name.upper()} {'='*45}")
    print(f"• Living Lexicon      : {total_words} words | Total Segments: {raw_segments} (Mean: {avg_len:.2f} phonemes/word)")
    print(f"• Consonant/Vowel     : {stats['cv_ratio']} (Normal Human Range: 0.75 — 2.20)")
    print(f"• Shannon Entropy     : {stats['entropy_evenness']:.3f} / 1.000")
    print(f"• Sample Living Word  : {sample_str}")
    print(f"• Active Inventories  : {len(c_all)} Consonants | {len(v_all)} Vowels")
    print(f"  ↳ Consonant Set     : {', '.join(c_all)}")
    print(f"  ↳ Vowel Set         : {', '.join(v_all)}")

    if clicks or ejectives or implosives or pharyngealized:
        layers = []
        if clicks: layers.append(f"Clicks: {', '.join(clicks)}")
        if ejectives: layers.append(f"Ejectives: {', '.join(ejectives)}")
        if implosives: layers.append(f"Implosives: {', '.join(implosives)}")
        if pharyngealized: layers.append(f"Pharyngealized: {', '.join(pharyngealized)}")
        print(f"• Non-Pulmonic Layers : {' | '.join(layers)}")

    if anomalies:
        print(f"• ⚠️  ANOMALY ALERTS  : {', '.join(anomalies)}")
    else:
        print(f"• ✓  Balance Status   : Healthy (Zero single-sound monopolies, balanced dispersion)")

    print("\n--- CONSONANT FREQUENCY BREAKDOWN (Top 10) ---")
    for rank, (ipa, pct) in enumerate(list(stats["consonants"].items())[:10], start=1):
        bar = "█" * int(round(pct * 1.5))
        print(f"  {rank:>2}. /{ipa:<4}/ : {pct:>5.1f}%  {bar}")

    print("\n--- VOWEL FREQUENCY BREAKDOWN ---")
    for rank, (ipa, pct) in enumerate(stats["vowels"].items(), start=1):
        bar = "█" * int(round(pct * 1.2))
        print(f"  {rank:>2}. /{ipa:<4}/ : {pct:>5.1f}%  {bar}")


def run_phoneme_statistics_test():
    print("=" * 115)
    print("  MULTI-BIOME PHONEME DISTRIBUTION & FREQUENCY STATISTICAL AUDIT (200 EPOCHS)")
    print("=" * 115)

    biomes = [
        ("Tropical-Rainforest", Environment(temperature=0.65, humidity=0.85, altitude=0.05, vegetation=0.80, ambient_noise=0.20, population=0.30)),
        ("Alpine-Highlands",    Environment(temperature=0.25, humidity=0.35, altitude=0.85, vegetation=0.15, ambient_noise=0.35, population=0.25)),
        ("Arid-Savannah",       Environment(temperature=0.80, humidity=0.15, altitude=0.20, vegetation=0.15, ambient_noise=0.15, population=0.20)),
        ("Temperate-Metropolis", Environment(temperature=0.50, humidity=0.50, altitude=0.15, vegetation=0.40, ambient_noise=0.45, population=0.90)),
    ]

    for name, env in biomes:
        print(f"\n>> Simulating {name} (T={env.temperature:.2f}, H={env.humidity:.2f}, Alt={env.altitude:.2f}, Pop={env.population:.2f})...")
        lang = Language(name, environment=env)
        populate_proto_phonemes(lang)
        lang.generate_lexicon()

        for ep in range(1, 201):
            lang.evolve(reduction_strength=0.75, enable_neologisms=True, current_generation=ep)

        print_language_stats(lang)

    print("\n" + "=" * 115)
    print("✓ All multi-biome statistical audits complete!")


if __name__ == "__main__":
    run_phoneme_statistics_test()

  MULTI-BIOME PHONEME DISTRIBUTION & FREQUENCY STATISTICAL AUDIT (200 EPOCHS)

>> Simulating Tropical-Rainforest (T=0.65, H=0.85, Alt=0.05, Pop=0.30)...

============================================= TROPICAL-RAINFOREST =============================================
• Living Lexicon      : 1524 words | Total Segments: 8438 (Mean: 5.54 phonemes/word)
• Consonant/Vowel     : 0.85 (Normal Human Range: 0.75 — 2.20)
• Shannon Entropy     : 0.739 / 1.000
• Sample Living Word  : /ɓe˩ɜ˧wo̞˧˥jɜ˩/ (#2096)
• Active Inventories  : 17 Consonants | 8 Vowels
  ↳ Consonant Set     : b, d, d͡z, j, l, m, n, p, r, t, t͡ʃ, v, w, z, ɗ, ɡ̆, ʟ̝
  ↳ Vowel Set         : e, i, o, u, ä, ɐ, ɔ, ɜ
• Non-Pulmonic Layers : Implosives: ɗ
• ✓  Balance Status   : Healthy (Zero single-sound monopolies, balanced dispersion)

--- CONSONANT FREQUENCY BREAKDOWN (Top 10) ---
   1. /d   / :   8.2%  ████████████
   2. /r   / :   8.1%  ████████████
   3. /z   / :   7.9%  ████████████
   4. /t   / :   7.8%  ████████████
   5. /n  

KeyboardInterrupt: 

In [1]:
"""
test_italian_expansion.py — 3-Stage Evolutionary Simulation of an Italic Population:
1. Archaic Italic / Pastoral Roots (Epochs 1 - 150)
2. Imperial Expansion & Pan-Mediterranean Trade (Epochs 151 - 300)
3. Modern Italian Standardization (Epochs 301 - 450)
"""

from __future__ import annotations

import math
from phoneme import Phoneme, PhonemeKind
from language import Language
from world_map import Environment


def populate_proto_phonemes(lang: Language):
    """Initializes standard cardinal vowels and baseline consonants."""
    for pt in [(6, 0, 0), (6, 2, 1), (4, 0, 0), (4, 2, 1), (0, 1, 0)]:  # /i, u, e, o, a/
        lang.add_phoneme(Phoneme(PhonemeKind.VOWEL, pt))

    for pt in [
        (0, 0, 0), (0, 0, 1), (3, 0, 0), (3, 0, 1), (7, 0, 0), (7, 0, 1),   # p, b, t, d, k, g
        (1, 4, 0), (1, 4, 1), (3, 4, 0), (3, 4, 1), (4, 4, 0), (7, 4, 0), (10, 4, 0),  # f, v, s, z, ʃ, x, h
        (0, 1, 1), (3, 1, 1),                                                 # m, n
        (3, 7, 1), (0, 7, 1)                                                  # l, w
    ]:
        lang.add_phoneme(Phoneme(PhonemeKind.CONSONANT, pt))


def print_epoch_phoneme_snapshot(lang: Language, epoch: int):
    """Prints a detailed phonological and lexical snapshot."""
    stats = lang.get_phoneme_distribution_stats()
    total_words = len(lang.words)
    raw_segments = stats.get("raw_total_segments", 0)
    avg_len = raw_segments / max(1, total_words)

    v_all = sorted(list(set(p.ipa for p in lang.vowels)))
    v_oral = [v for v in v_all if "\u0303" not in v]
    v_nasal = [v for v in v_all if "\u0303" in v]

    c_all = sorted(list(set(p.ipa for p in lang.consonants)))
    affricates = [c for c in c_all if "͡" in c or "͜" in c]
    clicks = [c for c in c_all if any(sym in c for sym in ('ǃ', 'ʘ', 'ǂ', 'ǁ', 'ǀ'))]
    ejectives = [c for c in c_all if "ʼ" in c and not any(sym in c for sym in ('ǃ', 'ʘ', 'ǂ', 'ǁ', 'ǀ'))]
    implosives = [c for c in c_all if any(sym in c for sym in ('ɓ', 'ɗ', 'ɠ', 'ʄ', 'ʛ'))]

    # Sample living word
    sample_words = list(lang.words.values())
    sample_w = sample_words[len(sample_words) // 2] if sample_words else None
    sample_str = f"/{sample_w.form}/ (#{sample_w.id})" if sample_w else "N/A"

    print("=" * 110)
    print(f"  EPOCH {epoch:>3} / 450 SNAPSHOT — {lang.name.upper()}")
    print("=" * 110)
    print(f"• Living Lexicon      : {total_words} words | Total Segments: {raw_segments} (Mean: {avg_len:.2f} phonemes/word)")
    print(f"• Consonant/Vowel     : {stats['cv_ratio']} (Mediterranean Target: 1.40 — 1.60)")
    print(f"• Shannon Entropy     : {stats['entropy_evenness']:.3f} / 1.000")
    print(f"• Sample Living Word  : {sample_str}")
    print("-" * 110)

    v_nasal_str = f" | Nasalized ({len(v_nasal)}): {', '.join(v_nasal)}" if v_nasal else ""
    print(f"• Vowels ({len(v_all)} Total) : Oral ({len(v_oral)}): {', '.join(v_oral)}{v_nasal_str}")
    print(f"• Consonants ({len(c_all)} Total): {', '.join(c_all)}")

    if affricates:
        print(f"• Sibilant Affricates : {', '.join(affricates)}")
    if clicks or ejectives or implosives:
        layers = []
        if clicks: layers.append(f"Clicks: {', '.join(clicks)}")
        if ejectives: layers.append(f"Ejectives: {', '.join(ejectives)}")
        if implosives: layers.append(f"Implosives: {', '.join(implosives)}")
        print(f"• Non-Pulmonic Layers : {' | '.join(layers)}")

    top_c = stats.get("top_5_consonants", [])
    top_v = stats.get("top_5_vowels", list(stats.get("vowels", {}).items())[:5])
    top_c_str = ", ".join([f"/{ipa}/ ({pct:.1f}%)" for ipa, pct in top_c])
    top_v_str = ", ".join([f"/{ipa}/ ({pct:.1f}%)" for ipa, pct in top_v])

    print(f"• Top Consonants     : {top_c_str}")
    print(f"• Top Vowels         : {top_v_str}")
    print("")

    # Extract 7D Cultural Genome
    attn = lang.cultural_attention
    dim_names = ["Concreteness", "Animacy", "Valence", "Potency", "Dynamism", "Sociality", "Extension"]
    
    # Identify Cultural Mindset (Top 2 dimensions)
    ranked_dims = sorted(zip(dim_names, attn), key=lambda x: x[1], reverse=True)
    mindset_str = f"{ranked_dims[0][0]} + {ranked_dims[1][0]}"
    
    print(f"• Cultural Mindset    : [{mindset_str}] | Inertia (Purism): {lang.cultural_inertia:.2f}")
    print(f"• 7D Cultural Genome  : C:{attn[0]:.2f} | A:{attn[1]:.2f} | V:{attn[2]:.2f} | P:{attn[3]:.2f} | D:{attn[4]:.2f} | S:{attn[5]:.2f} | E:{attn[6]:.2f}")


def run_italic_expansion_simulation():
    # =========================================================================
    # STAGE 1: ARCHAIC ITALIC / PASTORAL ROOTS (Epochs 1 - 150)
    # =========================================================================
    env_archaic = Environment(
        temperature=0.55,
        humidity=0.58,
        altitude=0.18,
        vegetation=0.45,
        ambient_noise=0.25,
        population=0.35,  # Moderate agricultural community
    )

    profile_archaic = env_archaic.generate_profile(
        name="Italic-Archaic Profile",
        initial_head_directionality=0.45,  # Flexible SVO / Penultimate Stress
    )

    lang = Language("Romano-Italic", profile=profile_archaic, environment=env_archaic)
    populate_proto_phonemes(lang)
    lang.generate_lexicon()

    print("\n" + "#" * 110)
    print("  STAGE 1: ARCHAIC ITALIC / PASTORAL ROOTS (EPOCHS 1 — 150)")
    print("  Environment: T=0.55, H=0.58, Alt=0.18, Veg=0.45, Noise=0.25, Pop=0.35")
    print("#" * 110)

    print_epoch_phoneme_snapshot(lang, epoch=0)

    for ep in range(1, 151):
        lang.evolve(reduction_strength=0.75, enable_neologisms=True, current_generation=ep)
        if ep in (50, 100, 150):
            print_epoch_phoneme_snapshot(lang, epoch=ep)

    # =========================================================================
    # STAGE 2: IMPERIAL EXPANSION & MEDITERRANEAN TRADE (Epochs 151 - 300)
    # =========================================================================
    env_expansion = Environment(
        temperature=0.58,
        humidity=0.54,
        altitude=0.12,
        vegetation=0.35,
        ambient_noise=0.40,  # Coastal trade hubs, urbanization
        population=0.85,     # Large empire population (Demographic leveling active)
    )

    lang.environment = env_expansion
    lang.profile = env_expansion.generate_profile(
        name="Italic-Imperial Profile",
        initial_head_directionality=0.50,  # Balanced SVO syntax
    )

    print("\n" + "#" * 110)
    print("  STAGE 2: IMPERIAL EXPANSION & MEDITERRANEAN TRADE (EPOCHS 151 — 300)")
    print("  Environment: T=0.58, H=0.54, Alt=0.12, Veg=0.35, Noise=0.40, Pop=0.85")
    print("#" * 110)

    for ep in range(151, 301):
        lang.evolve(reduction_strength=0.75, enable_neologisms=True, current_generation=ep)
        if ep in (200, 250, 300):
            print_epoch_phoneme_snapshot(lang, epoch=ep)

    # =========================================================================
    # STAGE 3: MODERN ITALIAN STANDARDIZATION (Epochs 301 - 450)
    # =========================================================================
    env_modern = Environment(
        temperature=0.60,
        humidity=0.52,
        altitude=0.10,
        vegetation=0.30,
        ambient_noise=0.45,  # Modern urban acoustic noise floor
        population=0.92,     # Dense metropolitan standardization
    )

    lang.environment = env_modern
    lang.profile = env_modern.generate_profile(
        name="Italic-Modern Profile",
        initial_head_directionality=0.52,  # Canonical SVO
    )

    print("\n" + "#" * 110)
    print("  STAGE 3: MODERN ITALIAN STANDARDIZATION (EPOCHS 301 — 450)")
    print("  Environment: T=0.60, H=0.52, Alt=0.10, Veg=0.30, Noise=0.45, Pop=0.92")
    print("#" * 110)

    for ep in range(301, 451):
        lang.evolve(reduction_strength=0.75, enable_neologisms=True, current_generation=ep)
        if ep in (350, 400, 450):
            print_epoch_phoneme_snapshot(lang, epoch=ep)


if __name__ == "__main__":
    run_italic_expansion_simulation()


##############################################################################################################
  STAGE 1: ARCHAIC ITALIC / PASTORAL ROOTS (EPOCHS 1 — 150)
  Environment: T=0.55, H=0.58, Alt=0.18, Veg=0.45, Noise=0.25, Pop=0.35
##############################################################################################################
  EPOCH   0 / 450 SNAPSHOT — ROMANO-ITALIC
• Living Lexicon      : 61 words | Total Segments: 280 (Mean: 4.59 phonemes/word)
• Consonant/Vowel     : 1.14 (Mediterranean Target: 1.40 — 1.60)
• Shannon Entropy     : 0.890 / 1.000
• Sample Living Word  : /so˥˩ni˥/ (#1031)
--------------------------------------------------------------------------------------------------------------
• Vowels (5 Total) : Oral (5): e, i, o, u, ä
• Consonants (15 Total): b, d, f, h, k, l, m, n, p, t, w, z, ɡ, ɣ, ʃ
• Top Consonants     : /ʃ/ (14.2%), /p/ (13.9%), /k/ (9.2%), /l/ (9.1%), /n/ (8.1%)
• Top Vowels         : /o/ (29.1%), /e/ (27.0%), /ä/ (21.4%), /i/ 

In [ ]:
import visualizer
visualizer.start_server()


  EVOLANG REAL-TIME VISUALIZER RUNNING
  Open your browser at: http://localhost:8000



127.0.0.1 - - [10/Sep/2026 17:00:21] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Sep/2026 17:00:21] "GET /api/state HTTP/1.1" 200 -
127.0.0.1 - - [10/Sep/2026 17:01:50] "POST /api/step HTTP/1.1" 200 -
127.0.0.1 - - [10/Sep/2026 17:01:53] "POST /api/step HTTP/1.1" 200 -
